# 11 · Hyperparameter search with Ray Tune

The loop is a *qualitative* search (change the method); **Ray Tune** is a
*quantitative* one (dial in the hyperparameters). Each trial trains under budget
and returns the **evaluator's official score** — Tune never touches the test
split directly. We run a small ASHA search.

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from harness.data import make_synthetic_dataset, DatasetSpec, load_split, class_names

DATA = Path("../data/bdd-tiny.lance")
if not DATA.exists():
    make_synthetic_dataset(DATA, DatasetSpec(n=3000, seed=7))
print("dataset:", DATA, "| NOTE: these chapters scale to bdd-small/full; here we")
print("demonstrate the mechanics on the tiny tier so they run with no GPU/cluster.")

dataset: ../data/bdd-tiny.lance | NOTE: these chapters scale to bdd-small/full; here we
demonstrate the mechanics on the tiny tier so they run with no GPU/cluster.


In [2]:
try:
    import ray
    from ray import tune
    from ray.tune.schedulers import ASHAScheduler
    HAVE_RAY = True
except Exception as e:
    HAVE_RAY = False; print("Ray Tune not installed -> code only:", e)

2026-05-25 16:14:12,079	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


2026-05-25 16:14:12,425	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [3]:
def trainable(params):
    # Ray Tune runs each trial in its own working dir, so the dataset path must
    # be absolute (passed in via params["data"]).
    from harness.config import load_config
    from harness.train import train
    from harness.evaluator import evaluate
    from harness.mining import compute_sample_weights
    p = params["data"]
    cfg = load_config("base", {
        "dataset": {"path": p},
        "train": {"lr": params["lr"]},
        "model": {"width": params["width"]},
        "mining": {"enabled": True, "strategy": "fog_boost", "boost": params["boost"]},
        "budget": {"max_epochs": 8, "max_seconds": 60},
    })
    w = compute_sample_weights(p, cfg)
    m = train(cfg, p, sample_weights=w)
    return {"score": evaluate(m, p)["score"]}

if HAVE_RAY:
    if not ray.is_initialized():
        ray.init(num_cpus=4, logging_level="ERROR", include_dashboard=False)
    space = {"data": str(DATA.resolve()),
             "lr": tune.loguniform(3e-4, 3e-3),
             "width": tune.choice([16, 24, 32]),
             "boost": tune.uniform(3.0, 8.0)}
    tuner = tune.Tuner(
        tune.with_resources(trainable, {"cpu": 2}),
        param_space=space,
        tune_config=tune.TuneConfig(metric="score", mode="max",
                                    scheduler=ASHAScheduler(), num_samples=6))
    res = tuner.fit()
    best = res.get_best_result(metric="score", mode="max")
    print("best score:", round(best.metrics["score"], 4), "| config:", best.config)
else:
    print("(install ray[tune] to run this)")

best score: 1.0697 | config: {'data': '/home/user/insurance-agent/self-improving-ml-harness/data/bdd-tiny.lance', 'lr': 0.001412335445542304, 'width': 32, 'boost': 6.6964718765288005}


ASHA started 6 trials cheap and gave budget to the promising ones. The best
config is a candidate the *loop* can adopt — qualitative and quantitative search
compose, both judged by the same immutable score.